# Unsupervised learning: structure before labels

**Lecture 16 · Notebook 00 · CMOR 438 / INDE 577**

Supervised learning begins with a target. Unsupervised learning begins
with observations or relations and a more dangerous question:

> **What structure is present, and which parts of that structure are
> consequences of our representation and assumptions?**

We will use real wine-chemistry measurements to investigate three major
families: **clustering**, **dimensionality reduction**, and **graph or
relational learning**. Labels will be hidden while methods learn and
revealed only for an external scientific audit.

## How to use this notebook

Run the notebook from top to bottom with the **Rice DSM** kernel. Before
each figure, predict what you expect to see. Afterward, distinguish an
observed pattern from an explanation of that pattern.

**Core route (about 85 minutes):** task contract → database inspection →
scaling → clustering → dimensionality reduction → relational graph →
evaluation boundaries. **Practice and extension (about 35 minutes):**
sensitivity analysis, alternative neighborhood definitions, and a
contribution plan.

## Learning objectives

By the end, you should be able to:

1. distinguish supervised, unsupervised, and self-supervised objectives;
2. explain what clustering, dimensionality reduction, and graph methods return;
3. inspect a real dataset before choosing a representation or distance;
4. explain why feature scaling changes Euclidean geometry;
5. evaluate stability and use labels only as an external audit; and
6. state why an unsupervised pattern is not automatically a scientific discovery.

## Why this matters

Labels are scarce, delayed, disputed, or impossible in many scientific
and industrial settings. Researchers explore cell populations before
phenotypes are settled; engineers inspect customer behavior before an
outcome matures; astronomers search spectra for unusual populations;
security teams look for novel activity rather than only known attacks.

Unsupervised learning is powerful precisely because it can expose
structure not specified in advance. That freedom also makes it easy to
over-interpret a visually pleasing embedding or cluster assignment.

## Professional practice

A data scientist asks whether variables, distances, neighborhoods, and
validation evidence match the scientific question. A software engineer
asks which fitted transformation, schema, seed, and dependency produced
the artifact. Record both. A cluster identifier is an arbitrary code,
not an intrinsic name or causal category.

In [ ]:
# Load only local, versioned course data; no network is needed.
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import course_database_path

RANDOM_STATE = 438
DATABASE_PATH = course_database_path()
assert DATABASE_PATH.is_file(), "Run the course refresh command first."
sns.set_theme(style="whitegrid", context="notebook")
print("Database:", DATABASE_PATH)

## 1. Three kinds of unsupervised question

**Clustering** assigns or probabilistically associates observations with
groups under a stated similarity model. **Dimensionality reduction**
constructs a lower-dimensional representation that preserves a chosen
property such as variance, distance, neighborhood, or reconstruction.
**Graph and relational learning** treats connections as first-class
data: nodes may be molecules, people, papers, or machines; edges may be
bonds, interactions, citations, or transactions.

These families overlap. A graph can be embedded into vectors and then
clustered. A lower-dimensional representation can feed a supervised
model. The objective—not the plotting command—defines the task.

Self-supervised learning also uses unlabeled observations, but creates a
prediction problem from the data itself: mask a token, predict a view,
or contrast related and unrelated examples. Its learned representation
is then reused elsewhere.

## 2. Inspect before modeling

The UCI Wine data contain 178 chemical analyses from three cultivars.
We will query the measurements from SQLite, inspect units and ranges,
and remove the cultivar fields from the learning matrix. The cultivar
code is retained separately only to audit what the unlabeled analysis
recovers.

In [ ]:
# Inspect the repository's data catalog before reading observations.
with sqlite3.connect(DATABASE_PATH) as connection:
    catalog = pd.read_sql_query(
        "SELECT dataset_id, title, source_url, license_note FROM dataset_catalog",
        connection,
    )
    wine = pd.read_sql_query(
        "SELECT * FROM wine_observations ORDER BY observation_id",
        connection,
    )

display(catalog)
print("Shape:", wine.shape)
display(wine.head())
assert len(wine) == 178
assert wine["cultivar_code"].nunique() == 3

In [ ]:
# Separate identifiers and audit labels from the unlabeled feature matrix.
excluded = {"observation_id", "cultivar_code", "cultivar_label"}
feature_names = [column for column in wine.columns if column not in excluded]
wine_features = wine[feature_names]
audit_labels = wine["cultivar_code"].to_numpy()

summary = wine_features.agg(["mean", "std", "min", "max"]).T
summary["range"] = summary["max"] - summary["min"]
display(summary.sort_values("range", ascending=False))

assert wine_features.shape == (178, 13)
assert wine_features.notna().all().all()

### Why inspect ranges?

Euclidean distance squares coordinate differences. A feature measured
on a numerically large scale can dominate even when it is not more
scientifically important. Standardization replaces feature $j$ by

\[
z_{ij}=\frac{x_{ij}-\bar x_j}{s_j}.
\]

This does not make the features “unitless truth.” It encodes the policy
that a one-standard-deviation change is comparable across features.
Robust scaling, domain units, or a learned metric may be more defensible
under contamination, bounded variables, or known measurement error.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Compare geometry before and after the explicit scaling policy.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(wine_features)
raw_distance = np.linalg.norm(wine_features.iloc[0] - wine_features.iloc[1])
scaled_distance = np.linalg.norm(scaled_features[0] - scaled_features[1])

print(f"Raw Euclidean distance:    {raw_distance:.2f}")
print(f"Scaled Euclidean distance: {scaled_distance:.2f}")
assert np.allclose(scaled_features.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(scaled_features.std(axis=0), 1.0, atol=1e-12)

## 3. Worked example: clustering chemical profiles

$K$-means chooses centroids $\mu_1,\ldots,\mu_K$ and assignments
$c_i$ to minimize within-cluster squared distance:

\[
\min_{c,\mu}\sum_{i=1}^{n}\lVert z_i-\mu_{c_i}\rVert_2^2.
\]

Alternating assignment and centroid updates decreases this objective,
but need not find its global minimum. The value of $K$, scaling,
initialization, outliers, and the assumption of roughly compact groups
all matter. Cluster number 0 has no stable semantic meaning.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Fit without cultivar labels. Multiple initializations reduce dependence
# on one unlucky starting configuration.
kmeans = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE)
cluster_codes = kmeans.fit_predict(scaled_features)

silhouette = silhouette_score(scaled_features, cluster_codes)
external_ari = adjusted_rand_score(audit_labels, cluster_codes)
print(f"Silhouette (internal geometry): {silhouette:.3f}")
print(f"Adjusted Rand index (label audit): {external_ari:.3f}")
display(pd.crosstab(audit_labels, cluster_codes, rownames=["cultivar"],
                    colnames=["cluster"]))

assert cluster_codes.shape == (178,)
assert set(np.unique(cluster_codes)) == {0, 1, 2}
assert -1.0 <= external_ari <= 1.0

The silhouette compares each observation's average within-cluster
distance with its nearest alternative cluster. It audits separation
under the same geometry used by the method; it cannot prove the groups
are scientifically real. The adjusted Rand index compares pairs of
assignments against labels while correcting for chance. Because the
labels were not used to fit or choose this model, they provide an
external audit—not an unsupervised training signal.

## 4. Dimensionality reduction as a view, not a verdict

Thirteen coordinates are difficult to inspect. Principal component
analysis (PCA) finds orthogonal directions of decreasing sample
variance. We use two components for a view of the fitted clusters. A
two-dimensional projection inevitably discards information, so apparent
overlap or separation in the plot need not match the full space.

In [ ]:
from sklearn.decomposition import PCA

# PCA learns directions from features only; colors are added after fitting.
overview_pca = PCA(n_components=2)
wine_coordinates = overview_pca.fit_transform(scaled_features)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
sns.scatterplot(x=wine_coordinates[:, 0], y=wine_coordinates[:, 1],
                hue=cluster_codes, palette="deep", ax=axes[0], legend="full")
axes[0].set(title="Unsupervised K-means assignments",
            xlabel="principal component 1", ylabel="principal component 2")
sns.scatterplot(x=wine_coordinates[:, 0], y=wine_coordinates[:, 1],
                hue=audit_labels, palette="colorblind", ax=axes[1], legend="full")
axes[1].set(title="Cultivar labels revealed only for audit",
            xlabel="principal component 1", ylabel="principal component 2")
plt.show()

print("Variance visible in two components:",
      f"{overview_pca.explained_variance_ratio_.sum():.1%}")
assert wine_coordinates.shape == (178, 2)

## 5. Graph and relational data

A feature table describes each observation independently. A graph
$G=(V,E)$ represents entities and relations. Its adjacency matrix $A$
has $A_{ij}\ne0$ when node $i$ relates to node $j$; edge values may
encode distance, strength, direction, time, or type.

Native graph examples include molecular bonds, power-grid connections,
supply chains, citations, and protein interactions. Here we construct a
**similarity graph** from the wine table: connect each observation to
nearby chemical profiles. This construction is a modeling decision. A
different metric or neighbor count produces a different graph.

In [ ]:
from sklearn.neighbors import kneighbors_graph

# Build a symmetric five-nearest-neighbor graph in standardized space.
directed_graph = kneighbors_graph(
    scaled_features, n_neighbors=5, mode="connectivity", include_self=False
)
adjacency = directed_graph.maximum(directed_graph.T).toarray()
degrees = adjacency.sum(axis=1)

print("Nodes:", adjacency.shape[0])
print("Undirected edges:", int(adjacency.sum() / 2))
print("Degree range:", int(degrees.min()), "to", int(degrees.max()))
assert np.allclose(adjacency, adjacency.T)
assert np.all(np.diag(adjacency) == 0)
assert degrees.min() >= 5

In [ ]:
# Draw the relational view using the PCA coordinates only as page positions.
fig, ax = plt.subplots(figsize=(9, 7))
rows, columns = np.where(np.triu(adjacency, k=1) > 0)
for source, target in zip(rows, columns, strict=True):
    ax.plot(wine_coordinates[[source, target], 0],
            wine_coordinates[[source, target], 1],
            color="#9aa6b2", alpha=0.12, linewidth=0.7, zorder=1)
points = ax.scatter(wine_coordinates[:, 0], wine_coordinates[:, 1],
                    c=audit_labels, cmap="viridis", s=28 + 4 * degrees,
                    edgecolor="white", linewidth=0.4, zorder=2)
ax.set(title="Chemical-similarity graph: node size is degree",
       xlabel="PCA coordinate 1 (layout only)",
       ylabel="PCA coordinate 2 (layout only)")
fig.colorbar(points, ax=ax, label="cultivar audit code")
plt.show()

### What a graph changes

The graph permits questions the table alone does not state explicitly:
Which nodes bridge regions? Are there communities? Does information
diffuse? Which links are surprising? Spectral embeddings use eigenvectors
of an adjacency- or Laplacian-derived operator; graph neural networks
repeatedly aggregate neighbor information. Neither makes relations
causal. Missing or biased edges can dominate the result.

**Common failure:** interpreting the PCA page positions as the graph
algorithm. Here the edges were defined in all 13 standardized features;
PCA only supplies readable plotting coordinates.

## Guided practice: sensitivity is evidence

Refit $K$-means for $K=2,\ldots,8$. Record inertia and silhouette. Then
repeat after removing scaling. Success means you can explain why inertia
cannot select $K$ by simply choosing its minimum and why the raw-feature
result is governed by a different geometry.

In [ ]:
# Build a compact sensitivity table rather than trusting one configuration.
sensitivity_rows = []
for candidate_clusters in range(2, 9):
    candidate = KMeans(n_clusters=candidate_clusters, n_init=20,
                       random_state=RANDOM_STATE)
    assignments = candidate.fit_predict(scaled_features)
    sensitivity_rows.append({
        "clusters": candidate_clusters,
        "inertia": candidate.inertia_,
        "silhouette": silhouette_score(scaled_features, assignments),
    })
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity)
assert sensitivity["inertia"].is_monotonic_decreasing
assert sensitivity.loc[sensitivity["silhouette"].idxmax(), "clusters"] >= 2

## Independent practice

Change the graph neighborhood size from 5 to 3, 10, and 20. For each
graph, report edge count, isolated nodes, and degree range. Decide which
construction is useful for investigating local cultivar mixing. Success
requires a written criterion; “the plot looks nicest” is not sufficient.

## Extension: from laboratory to durable contribution

Propose one reusable unsupervised component for the course package or
companion student repository. Put its implementation under `src/`, its
mathematical and failure contracts under `tests/`, and keep this
notebook focused on evidence and interpretation. Your PR plan must name:

- input shape, dtype, missing-value, and fitted-state behavior;
- randomness and convergence policy;
- at least one analytic invariant and trusted-library comparison;
- a lawful small fixture and a real-data application; and
- one limitation that the API cannot prevent.

## Debugging and common failure modes

1. **Clusters change between runs:** fix and record initialization; then
   measure stability rather than hiding it.
2. **One feature dominates:** inspect units/ranges and justify scaling.
3. **A beautiful 2-D plot disagrees with metrics:** quantify how much the
   projection retained and evaluate in the space where the model fit.
4. **Labels quietly influence preprocessing or $K$:** the procedure is no
   longer purely unsupervised; document the semi-supervised choice.
5. **Graph conclusions change with $k$:** neighborhood construction is a
   model, not clerical preprocessing.

Never rename clusters by majority label and then claim the method
discovered those classes without label information.

## Retrieval practice

1. What mathematical object does clustering return? PCA? A graph method?
2. Why does standardization change $K$-means?
3. Why are cluster identifiers arbitrary?
4. What does the silhouette measure, and what can it not establish?
5. When may labels be used to evaluate an unsupervised result?
6. Name two modeling choices hidden inside a nearest-neighbor graph.

## Takeaway

Unsupervised learning does not mean assumption-free learning. It replaces
a supplied target with choices about representation, geometry, structure,
and validation. The next notebook makes one such choice fully explicit:
PCA as an optimal linear reconstruction and variance-preserving map.

## Further reading

- [scikit-learn clustering guide](https://scikit-learn.org/stable/modules/clustering.html)
- [scikit-learn decomposition guide](https://scikit-learn.org/stable/modules/decomposition.html)
- [scikit-learn nearest-neighbor graphs](https://scikit-learn.org/stable/modules/neighbors.html)
- [Python data model](https://docs.python.org/3/reference/datamodel.html)